# 🔬 Notebook 3: Google Calendar — Deep Dives

## 🛠️ Setup

```bash
cd 06-system-designs/google-calendar
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


This notebook zooms into three of the trickiest algorithms in a calendar service:

1. **Recurrence expansion** — turning one RRULE into the list of concrete occurrences inside a query window.
2. **Exception handling** — moving or cancelling a *single instance* of a recurring event.
3. **Free/busy (availability)** — answering "when are all of these people free for 30 minutes?"

We show each as **bad practice → best practice** so the *why* is obvious.

## 1. Recurrence expansion

### ❌ Bad: materialize every future occurrence at write time

Imagine the user creates "Daily stand-up at 9am" with no end date. The "simple" approach is to insert rows for every day for, say, the next 5 years.

- 5 years × 365 days = 1,825 rows **per event**.
- Changing the time means updating 1,825 rows.
- Storage explodes; updates become transactional nightmares.

### ✅ Best: store the **rule**, expand on read

Store one row with `rrule='FREQ=DAILY;BYHOUR=9'`. When a client asks for a specific week, expand just those ~7 rows in memory.

Below is a toy DAILY expander (easy to follow) followed by the **real** way using `python-dateutil`, which supports full iCalendar RRULE semantics (BYDAY, BYMONTHDAY, UNTIL, COUNT, EXDATE, ...).

In [ ]:
# Toy DAILY expander -- good for intuition, bad for production
from datetime import datetime, timedelta, timezone

def expand_daily(start, interval_days, window_from, window_to):
    out, t = [], start
    while t < window_from:
        t += timedelta(days=interval_days)
    while t <= window_to:
        out.append(t)
        t += timedelta(days=interval_days)
    return out

start = datetime(2026, 1, 1, 9, tzinfo=timezone.utc)
for x in expand_daily(
    start, 2,
    datetime(2026, 1, 5, tzinfo=timezone.utc),
    datetime(2026, 1, 15, tzinfo=timezone.utc),
):
    print(x)


In [ ]:
# Production-grade: python-dateutil understands the iCalendar RFC 5545 RRULE
from dateutil.rrule import rrulestr
from datetime import datetime, timezone

# Every weekday at 9:00 UTC, starting Jan 5 2026, 10 occurrences
rule = rrulestr(
    "FREQ=WEEKLY;BYDAY=MO,TU,WE,TH,FR;COUNT=10",
    dtstart=datetime(2026, 1, 5, 9, tzinfo=timezone.utc),
)

# Ask only for a 1-week window
for dt in rule.between(
    datetime(2026, 1, 5, tzinfo=timezone.utc),
    datetime(2026, 1, 9, 23, 59, tzinfo=timezone.utc),
    inc=True,
):
    print(dt)


### ⏰ Where recurrence and timezones collide (the bug that ships)

Both cells above expanded a rule **in UTC**, which is fine only because the events were
defined in UTC. Real users define recurring events in *wall-clock* terms: "daily standup,
9:00am, my time."

If you store `starts_at` in UTC and expand the rule from there, every occurrence keeps a
constant **UTC offset**. Across a DST transition the wall-clock time therefore *moves*:
your 9:00am standup becomes 10:00am, for everyone, silently, on a Sunday night.

The fix: for a recurring event, the source of truth is **(local wall time, IANA zone)**.
Expand the rule over naive local times, then convert each occurrence to UTC — the offset
is looked up *per occurrence*, so it changes exactly when the zone says it should.

US DST starts Sunday **8 March 2026**. Let's schedule a 9am standup the Thursday before
and watch both versions cross it.

In [ ]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from dateutil.rrule import rrulestr

LA = ZoneInfo("America/Los_Angeles")
WALL_START = datetime(2026, 3, 5, 9, 0)          # naive: "9:00am, LA time"
RULE = "FREQ=DAILY;COUNT=6"

# 🚫 BAD -- pin to an instant, expand in UTC. The offset is frozen at PST (-08:00).
utc_anchor = WALL_START.replace(tzinfo=LA).astimezone(timezone.utc)
bad = [dt.astimezone(LA) for dt in rrulestr(RULE, dtstart=utc_anchor)]

# ✅ BEST -- expand over naive LOCAL times, attach the zone per occurrence, then go to UTC.
good = [dt.replace(tzinfo=LA) for dt in rrulestr(RULE, dtstart=WALL_START)]

print(f"{'date':<12} {'🚫 expanded in UTC':<38} {'✅ expanded in LA':<22}")
print("-" * 74)
for b, g in zip(bad, good):
    drift = "  ← DRIFTED" if b.strftime("%H:%M") != "09:00" else ""
    left  = f"{b:%H:%M %Z} = {b.astimezone(timezone.utc):%H:%MZ}{drift}"
    right = f"{g:%H:%M %Z} = {g.astimezone(timezone.utc):%H:%MZ}"
    print(f"{g:%Y-%m-%d}   {left:<38} {right:<22}")

assert {b.strftime("%H:%M") for b in bad} == {"09:00", "10:00"}, "UTC expansion must drift"
assert {g.strftime("%H:%M") for g in good} == {"09:00"}, "local expansion must stay at 9am"
print("\n🚫 BAD : the UTC instant is held constant, so the wall clock moves — 9am → 10am.")
print("✅ BEST: the wall clock is held constant, so the UTC instant moves — 17:00Z → 16:00Z.")
print("\nAcross a DST boundary you can hold exactly one of those two constant. A recurring")
print("event is a promise about the wall clock, so that is the one you must preserve.")

### The two nasty edge cases inside the transition

Expanding in local time is necessary but not sufficient — some wall-clock times are not
real, and some happen twice.

- **Spring forward**: 02:30 on 8 Mar 2026 in LA **does not exist**. A 2:30am daily event
  has nothing to map to that day.
- **Fall back**: 01:30 on 1 Nov 2026 in LA **happens twice** (once PDT, once PST). Which
  one did the user mean?

Python's `zoneinfo` encodes the answer in `fold`: `fold=0` is the first (earlier, DST)
occurrence, `fold=1` the second. RFC 5545 says to skip nonexistent times; most products
instead shift them forward. Whatever you pick, **pick it explicitly and write it down** —
the failure mode of leaving it to the library is that your reminder fires an hour early
for half your users, twice a year.

In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

LA = ZoneInfo("America/Los_Angeles")

def is_nonexistent(naive: datetime, tz) -> bool:
    """True if this wall-clock time is skipped by a spring-forward jump.
    Test: local → UTC → local does not round-trip."""
    aware = naive.replace(tzinfo=tz)
    return aware.astimezone(timezone.utc).astimezone(tz).replace(tzinfo=None) != naive

def is_ambiguous(naive: datetime, tz) -> bool:
    """True if this wall-clock time happens TWICE (fall-back).
    The two folds disagree on the offset -- but so does a gap, so exclude those."""
    differs = (naive.replace(tzinfo=tz, fold=0).utcoffset()
               != naive.replace(tzinfo=tz, fold=1).utcoffset())
    return differs and not is_nonexistent(naive, tz)

cases = [
    ("normal day", datetime(2026, 3,  5, 2, 30)),
    ("spring fwd", datetime(2026, 3,  8, 2, 30)),   # 2:30am does not exist that day
    ("fall back ", datetime(2026, 11, 1, 1, 30)),   # 1:30am happens twice that day
]
for label, naive in cases:
    gap, dup = is_nonexistent(naive, LA), is_ambiguous(naive, LA)
    print(f"{label}  {naive}  nonexistent={gap!s:<5} ambiguous={dup!s:<5}", end="  ")
    if gap:
        # Policy choice: shift forward past the gap. RFC 5545's alternative is to skip
        # the occurrence entirely. Either is defensible; silence is not.
        shifted = naive + timedelta(hours=1)
        print(f"→ no valid instant; policy = shift to {shifted:%H:%M} "
              f"({shifted.replace(tzinfo=LA).astimezone(timezone.utc):%H:%MZ})")
    elif dup:
        first  = naive.replace(tzinfo=LA, fold=0).astimezone(timezone.utc)
        second = naive.replace(tzinfo=LA, fold=1).astimezone(timezone.utc)
        print(f"→ {first:%H:%MZ} or {second:%H:%MZ} — an hour apart; "
              f"policy = fold=0 (the first one)")
    else:
        print(f"→ {naive.replace(tzinfo=LA).astimezone(timezone.utc):%H:%MZ}")

assert is_nonexistent(datetime(2026, 3, 8, 2, 30), LA) and not is_ambiguous(datetime(2026, 3, 8, 2, 30), LA)
assert is_ambiguous(datetime(2026, 11, 1, 1, 30), LA) and not is_nonexistent(datetime(2026, 11, 1, 1, 30), LA)
print("""
Two more consequences that are easy to miss:
  * A stored occurrence's UTC instant can change with NOBODY editing the event --
    a government moves a DST date, tzdata ships an update, and every future occurrence
    in that zone shifts. So never persist expanded UTC instants far ahead, and pin
    (and version) the tzdata release your data was computed against.
  * Cross-timezone invitees are not symmetric. Organizer in LA, invitee in Tokyo (no
    DST): after the US transition the meeting really does move an hour for Tokyo.
    That is correct behaviour -- surface it in the UI instead of hiding it.""")

## 2. Overriding or cancelling one occurrence

Users often want to tweak **just one instance** of a recurring event:

- "Move *next* Monday's 1:1 to Tuesday" (override).
- "Skip the 1:1 this week — I'm on vacation" (cancel).

### Data model

```
events          : (id, rrule, starts_at, ends_at, ...)
event_exceptions: (event_id, original_start, new_start NULLABLE, cancelled BOOL)
```

We keep the rule intact and store a small delta keyed by the occurrence's **original** start time. On read, we merge: for every expanded occurrence, look up an exception; if it's cancelled, drop it; if it has a `new_start`, replace it.

In [ ]:
from dateutil.rrule import rrulestr
from datetime import datetime, timedelta, timezone

rule = rrulestr(
    "FREQ=WEEKLY;BYDAY=MO;COUNT=8",
    dtstart=datetime(2026, 5, 4, 15, tzinfo=timezone.utc),
)

# Exception table (in-memory for the demo)
#   key   = original occurrence start (UTC)
#   value = None -> cancelled
#   value = dt   -> moved to this new start
exceptions = {}

def move(original_start, new_start):
    exceptions[original_start] = new_start

def cancel(original_start):
    exceptions[original_start] = None

def occurrences_in(window_from, window_to):
    result = []
    for original in rule.between(window_from, window_to, inc=True):
        if original in exceptions:
            ex = exceptions[original]
            if ex is None:
                continue
            result.append((original, ex))
        else:
            result.append((original, original))
    return result

move(datetime(2026, 5, 11, 15, tzinfo=timezone.utc),
     datetime(2026, 5, 12, 15, tzinfo=timezone.utc))
cancel(datetime(2026, 5, 18, 15, tzinfo=timezone.utc))

for original, shown in occurrences_in(
    datetime(2026, 5,  1, tzinfo=timezone.utc),
    datetime(2026, 6,  1, tzinfo=timezone.utc),
):
    tag = "(moved)" if original != shown else ""
    print(shown, tag)


**Why key on `original_start` and not on a synthetic `occurrence_id`?** Because when the base event's time shifts, the rule changes and synthetic IDs drift. The original start time is a stable identity for "this particular instance of the series".

## 3. Free/busy — the heart of availability

Question: *"Find me 30 minutes between 9am–5pm today when Alice, Bob, and Room 7 are **all** free."*

This is how "Find a time" and the room-booking suggester work.

### Mental model

Each attendee's calendar for the day is a set of **busy intervals**. An attendee is free precisely when they are NOT inside any busy interval. The whole meeting works only if *every* attendee is free at the same time.

So we need to:

1. Pull each attendee's busy intervals in the window.
2. **Merge** them into a single set of "someone is busy" intervals.
3. Walk through the gaps and return any gap ≥ the requested duration.

### ❌ Bad: minute-by-minute scan

Beginners often write this: split the window into 1-minute slots, mark each slot busy if *any* attendee is busy, then scan for runs of free minutes.

- For a 9-hour window that's 540 slots. With N attendees and M events per attendee it's **O(slots · N · M)**.
- Works for a demo, dies at scale (imagine scanning a year for 50 people).

In [ ]:
from datetime import datetime, timedelta, timezone

def find_slot_naive(busy_per_person, window_from, window_to, duration_min):
    total_minutes = int((window_to - window_from).total_seconds() // 60)
    busy_mask = [False] * total_minutes
    for busy_list in busy_per_person:
        for (s, e) in busy_list:
            i = max(0, int((s - window_from).total_seconds() // 60))
            j = min(total_minutes, int((e - window_from).total_seconds() // 60))
            for k in range(i, j):
                busy_mask[k] = True

    run = 0
    for idx, b in enumerate(busy_mask):
        run = 0 if b else run + 1
        if run >= duration_min:
            start_idx = idx - duration_min + 1
            return window_from + timedelta(minutes=start_idx)
    return None

W0 = datetime(2026, 5, 4,  9, tzinfo=timezone.utc)
W1 = datetime(2026, 5, 4, 17, tzinfo=timezone.utc)

alice = [(datetime(2026,5,4, 9,30,tzinfo=timezone.utc), datetime(2026,5,4,10,30,tzinfo=timezone.utc)),
         (datetime(2026,5,4,13, 0,tzinfo=timezone.utc), datetime(2026,5,4,14, 0,tzinfo=timezone.utc))]
bob   = [(datetime(2026,5,4,10, 0,tzinfo=timezone.utc), datetime(2026,5,4,11, 0,tzinfo=timezone.utc))]
room7 = [(datetime(2026,5,4,15, 0,tzinfo=timezone.utc), datetime(2026,5,4,16, 0,tzinfo=timezone.utc))]

print("First 30-min slot (naive):",
      find_slot_naive([alice, bob, room7], W0, W1, 30))


### ✅ Best: sweep-line / interval merge — O((N·M) log (N·M))

Classic trick: take all the busy intervals together, sort their endpoints, sweep left-to-right counting how many are "open". Whenever the count is zero, everyone is free — look at the gap between the last close and the next open.

This is the same algorithm as "merge overlapping intervals" from Leetcode, just with a min-duration filter at the end.

In [ ]:
from datetime import datetime, timedelta, timezone

def find_slot_sweep(busy_per_person, window_from, window_to, duration_min):
    events = []
    for busy_list in busy_per_person:
        for (s, e) in busy_list:
            s = max(s, window_from)
            e = min(e, window_to)
            if s < e:
                events.append((s, +1))
                events.append((e, -1))
    events.sort()

    duration = timedelta(minutes=duration_min)
    open_count = 0
    cursor = window_from
    for t, delta in events:
        if open_count == 0 and t - cursor >= duration:
            return cursor
        open_count += delta
        if open_count == 0:
            cursor = t
    if open_count == 0 and window_to - cursor >= duration:
        return cursor
    return None

W0 = datetime(2026, 5, 4,  9, tzinfo=timezone.utc)
W1 = datetime(2026, 5, 4, 17, tzinfo=timezone.utc)
alice = [(datetime(2026,5,4, 9,30,tzinfo=timezone.utc), datetime(2026,5,4,10,30,tzinfo=timezone.utc)),
         (datetime(2026,5,4,13, 0,tzinfo=timezone.utc), datetime(2026,5,4,14, 0,tzinfo=timezone.utc))]
bob   = [(datetime(2026,5,4,10, 0,tzinfo=timezone.utc), datetime(2026,5,4,11, 0,tzinfo=timezone.utc))]
room7 = [(datetime(2026,5,4,15, 0,tzinfo=timezone.utc), datetime(2026,5,4,16, 0,tzinfo=timezone.utc))]

print("First 30-min slot (sweep):",
      find_slot_sweep([alice, bob, room7], W0, W1, 30))


### Sanity check: both methods agree

A good habit when you replace a slow-but-obvious algorithm with a fast one is to **randomly compare** them on many inputs. If they ever disagree, you have a bug in the fast one.

In [ ]:
import random
from datetime import datetime, timedelta, timezone

random.seed(42)
W0 = datetime(2026, 5, 4, 9, tzinfo=timezone.utc)

def rand_busy():
    out = []
    for _ in range(random.randint(0, 3)):
        start_min = random.randint(0, 7*60)
        length    = random.randint(30, 120)
        s = W0 + timedelta(minutes=start_min)
        e = s + timedelta(minutes=length)
        out.append((s, e))
    return out

mismatches = 0
for _ in range(200):
    people = [rand_busy() for _ in range(random.randint(1, 4))]
    W1 = W0 + timedelta(hours=8)
    dur = random.choice([15, 30, 45, 60])
    a = find_slot_naive(people, W0, W1, dur)
    b = find_slot_sweep(people, W0, W1, dur)
    if a != b:
        mismatches += 1

print(f"mismatches across 200 random inputs: {mismatches}")
assert mismatches == 0, "sweep disagrees with naive -- bug!"


## Real-world caveats we glossed over

Good to know these exist even if we won't implement them here:

- **All-day events** (`DATE` not `DATE-TIME`) — spanning a whole day across timezones needs care.
- **EXDATE / RDATE** — RFC 5545 lets you exclude or inject individual dates into a series; `dateutil` supports both.
- **"This and future"** edits typically *split* the series: end the old rule with an `UNTIL`, create a new event starting at the modified occurrence.
- **Working hours & DND** — free/busy should subtract time outside the user's working hours and within Do-Not-Disturb windows.
- **Resource conflicts** — treating a room as "an attendee with a calendar" is the cleanest model; booking just adds an invitation for the room.
- **Delegation & ACLs** — Alice's assistant can act on her calendar; the permission check is on every read and write.
- **Reminder delivery at scale** — don't `SELECT ... WHERE fires_at <= now()` every second. Use a partitioned durable timer queue (see `reminder-alert` lab).


## Closing thoughts

- **Store rules, expand on read** — the core trick for recurring events.
- **Model exceptions as a small delta table** keyed by `original_start`.
- **Free/busy is interval-merge** — sweep-line beats minute-scanning by orders of magnitude.
- **UTC + IANA tz** — never forget which one is the source of truth.

If you internalize those four ideas, you've got the backbone of a calendar service.
